# 18 Kaggle - Clean Sparse ALT 8-channel CNN 601bp, two-snapshot temporal split

Bản sạch từ notebook 17, chỉ giữ luồng chính:

1. Đọc 2 snapshot ClinVar và FASTA GRCh38 từ `/kaggle/input`.
2. Lọc SNV GRCh38, nhãn Benign/Likely benign và Pathogenic/Likely pathogenic, REF match FASTA.
3. Build input 8-channel kiểu sparse-alt: `REF one-hot 4ch` + `ALT-at-center 4ch`.
4. Chia train/validation/test bằng true two-snapshot temporal split.
5. Train dilated CNN, chọn threshold trên validation, đánh giá test và lưu artifact.

Bản này không có negative controls, calibration, stratify diagnostics, extreme-error analysis hoặc các cell phân tích phụ.


In [ ]:
from pathlib import Path
import json
import os
import random
import time
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from IPython.display import display

try:
    from pyfaidx import Fasta
except ImportError as exc:
    raise ImportError("pyfaidx is required. On Kaggle, add/install pyfaidx before running this notebook.") from exc

IS_KAGGLE = Path("/kaggle/input").exists()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd() / "Data"
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "kaggle_work_temporal"
PROCESSED_DIR = WORK_DIR / "processed"
MODEL_DIR = WORK_DIR / "models"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LENGTH = 601
FLANK = LENGTH // 2
DATASET_TAG = "fixed_refseq_temporal_sparse_alt8"
LABEL_MODE = "all"  # all | definitive_only

SPLIT_MODES = ["two_snapshot_temporal"]
TEMPORAL_TRAIN_MAX_YEAR = 2022
TEMPORAL_VAL_YEAR = 2023
TEMPORAL_TEST_MIN_YEAR = 2024
DROP_MISSING_LAST_EVALUATED = True
EXACT_VARIANT_PURGE = True
PURGE_DISTANCE_BP = 5000
APPLY_COORDINATE_PURGE_TO_TIME_ONLY = False  # time_only đo riêng trục thời gian, không cắt 5kb theo genome block.
GENOME_BLOCK_SIZE_BP = 1_000_000
SEQUENCE_PURGE_MODE = "exact_ref"  # none | exact_ref | exact_tensor
MATCH_SPACE_ONLY_TRAIN_SIZE = True
SPACE_ONLY_TARGET_TRAIN_ROWS = None

POSITIONAL_ENCODING = "none"  # notebook 17: disabled, sparse-alt input is 8ch
POSITIONAL_DIM = 0
ARCHITECTURE = "dilated"
BATCH_SIZE = 256 if IS_KAGGLE else 64
EPOCHS = 5
HARD_NEGATIVE_EPOCHS = 2
LEARNING_RATE = 1e-3
HARD_NEGATIVE_LR = 3e-4
WEIGHT_DECAY = 1e-4
IMBALANCE_STRATEGY = "weighted_sampler"  # none | weighted_sampler | pos_weight | legacy_sqrt_sampler_pos_weight
RC_AUGMENT = True
RC_TTA = True
USE_AMP = True
NUM_WORKERS = 4 if IS_KAGGLE else 0
PREFETCH_FACTOR = 4
PERSISTENT_WORKERS = NUM_WORKERS > 0
RANDOM_STATE = 42

RUN_FILTER_RAW = True
RUN_BUILD_DATASET = True
RUN_SMOKE_TEST = True
RUN_MINI_TRAIN = False
RUN_FULL_TRAIN = True
DEBUG_MAX_TRAIN_ROWS = 2048
DEBUG_MAX_VAL_ROWS = 512
DEBUG_MAX_TEST_ROWS = 512

SAFE_BASE = (
    f"clean_sparse_alt_8ch_nopos_{LENGTH}_{DATASET_TAG}_{ARCHITECTURE}_"
    f"rcaug_rctta_imbalance_{IMBALANCE_STRATEGY}"
)

print("IS_KAGGLE:", IS_KAGGLE)
print("INPUT_ROOT:", INPUT_ROOT)
print("WORK_DIR:", WORK_DIR)
print("SPLIT_MODES:", SPLIT_MODES)


# === Two-snapshot temporal (notebook 16) ===
NEW_SNAPSHOT_NAME = "variant_summary_2026-05.txt"   # superset -> build dataset từ đây
OLD_SNAPSHOT_NAME = "variant_summary_2022-12.txt"   # membership -> train/val eligible
TWO_SNAPSHOT_VAL_FRACTION = 0.15                     # tách VAL khỏi tập OLD
TWO_SNAPSHOT_DEV_SPLIT = "genome_block"              # genome_block | gene_group
SPLIT_MODES = ["two_snapshot_temporal"]


In [ ]:
CHROMOSOME_TO_REFSEQ = {
    "1": "NC_000001.11", "2": "NC_000002.12", "3": "NC_000003.12", "4": "NC_000004.12",
    "5": "NC_000005.10", "6": "NC_000006.12", "7": "NC_000007.14", "8": "NC_000008.11",
    "9": "NC_000009.12", "10": "NC_000010.11", "11": "NC_000011.10", "12": "NC_000012.12",
    "13": "NC_000013.11", "14": "NC_000014.9", "15": "NC_000015.10", "16": "NC_000016.10",
    "17": "NC_000017.11", "18": "NC_000018.10", "19": "NC_000019.10", "20": "NC_000020.11",
    "21": "NC_000021.9", "22": "NC_000022.11", "X": "NC_000023.11", "Y": "NC_000024.10",
    "MT": "NC_012920.1", "M": "NC_012920.1",
}
POSITIVE_LABELS = {"Pathogenic", "Likely pathogenic"}
NEGATIVE_LABELS = {"Benign", "Likely benign"}
TRUSTED_REVIEW_PATTERN = "criteria provided|reviewed by expert panel|practice guideline"
LOW_CONFIDENCE_REVIEW = "no assertion criteria provided"
BASE_TO_INDEX = np.full(256, 255, dtype=np.uint8)
for _idx, _base in enumerate(b"ACGT"):
    BASE_TO_INDEX[_base] = _idx
COMPLEMENT = np.array([3, 2, 1, 0], dtype=np.int64)


def split_clinical_significance(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("/") if part.strip()]


def map_clean_label(value):
    labels = split_clinical_significance(value)
    label_set = set(labels)
    if not labels:
        return None
    if label_set <= POSITIVE_LABELS:
        return 1
    if label_set <= NEGATIVE_LABELS:
        return 0
    return None


def normalize_chromosome(chromosome):
    chrom = str(chromosome).strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    if chrom in {"M", "MT", "m", "mt"}:
        return "MT"
    return chrom.upper()


def clean_base(value):
    return str(value).strip().upper()


def choose_chromosome_fasta(row, fasta_names):
    chrom = normalize_chromosome(row["Chromosome"])
    accession = str(row.get("ChromosomeAccession", "")).strip()
    candidates = [accession, str(row["Chromosome"]).strip(), chrom, f"chr{chrom}", CHROMOSOME_TO_REFSEQ.get(chrom, "")]
    for candidate in candidates:
        if candidate and candidate in fasta_names:
            return candidate
    return ""


# ============================================================
# PATH TỚI INPUT — chạy được cả trên Kaggle lẫn local.
# Kaggle: dataset tự giải nén vào /kaggle/input/<slug>/ (2 file dated là THƯ MỤC chứa variant_summary.txt).
# Local : đọc từ ./Data.
# ============================================================
if IS_KAGGLE:
    _roots = sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir())
    print("Datasets trong /kaggle/input:", [p.name for p in _roots])
    KAGGLE_INPUT = Path("/kaggle/input/biodemo")          # sửa slug nếu khác
    if not KAGGLE_INPUT.exists() and len(_roots) == 1:
        KAGGLE_INPUT = _roots[0]                           # chỉ có 1 dataset -> tự lấy
    NEW_SNAPSHOT_PATH = KAGGLE_INPUT / "variant_summary_2026-05.txt" / "variant_summary.txt"
    OLD_SNAPSHOT_PATH = KAGGLE_INPUT / "variant_summary_2022-12.txt" / "variant_summary.txt"
    FASTA_PATH        = KAGGLE_INPUT / "ncbi_dataset" / "ncbi_dataset" / "data" / "GCF_000001405.26" / "GCF_000001405.26_GRCh38_genomic.fna"
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = PROJECT_DIR / "Data"
    NEW_SNAPSHOT_PATH = DATA_DIR / "variant_summary_2026-05.txt"
    OLD_SNAPSHOT_PATH = DATA_DIR / "variant_summary_2022-12.txt"
    FASTA_PATH        = DATA_DIR / "ncbi_dataset" / "ncbi_dataset" / "data" / "GCF_000001405.26" / "GCF_000001405.26_GRCh38_genomic.fna"

VARIANT_SUMMARY_PATH = NEW_SNAPSHOT_PATH   # build dataset từ snapshot MỚI (superset)

for _label, _p in [("NEW snapshot", NEW_SNAPSHOT_PATH), ("OLD snapshot", OLD_SNAPSHOT_PATH), ("FASTA", FASTA_PATH)]:
    assert Path(_p).exists(), f"Khong thay {_label}: {_p}\\nSua path o cell nay cho khop /kaggle/input/<slug>/."
    print(f"{_label:13s}: {_p}")

FILTERED_PATH = PROCESSED_DIR / "clinvar_filtered_step8_fixed_refseq_temporal.parquet"
METADATA_PATH = PROCESSED_DIR / f"clinvar_training_metadata_{LENGTH}_{DATASET_TAG}.parquet"
X_PATH = PROCESSED_DIR / f"X_ref_index_{LENGTH}_{DATASET_TAG}.npy"
Y_PATH = PROCESSED_DIR / f"y_{LENGTH}_{DATASET_TAG}.npy"
for name, path in [("filtered", FILTERED_PATH), ("metadata", METADATA_PATH), ("X", X_PATH), ("y", Y_PATH)]:
    print(f"{name:10s}", path, "exists=" + str(path.exists()))


In [ ]:
def filter_raw_clinvar():
    if FILTERED_PATH.exists() and not RUN_FILTER_RAW:
        print("Use existing filtered parquet:", FILTERED_PATH)
        return pd.read_parquet(FILTERED_PATH)

    genome = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True, rebuild=False)
    fasta_names = set(genome.keys())
    missing_primary = {chrom: acc for chrom, acc in CHROMOSOME_TO_REFSEQ.items() if acc not in fasta_names}
    if missing_primary:
        raise RuntimeError(f"Missing primary RefSeq accessions in FASTA: {missing_primary}")

    usecols = [
        "Type", "GeneSymbol", "ClinicalSignificance", "LastEvaluated", "Assembly",
        "ChromosomeAccession", "Chromosome", "ReviewStatus", "PositionVCF",
        "ReferenceAlleleVCF", "AlternateAlleleVCF",
    ]
    counters = Counter()
    kept_chunks = []
    reader = pd.read_csv(VARIANT_SUMMARY_PATH, sep="\t", usecols=usecols, chunksize=500_000, low_memory=False)

    for chunk in tqdm(reader, desc="Filter ClinVar chunks"):
        counters["raw_rows"] += len(chunk)
        chunk = chunk.copy()
        chunk["Y"] = chunk["ClinicalSignificance"].map(map_clean_label)
        mask = chunk["Y"].notna()
        counters["labeled_rows"] += int(mask.sum())
        mask &= chunk["Assembly"].eq("GRCh38")
        counters["grch38_labeled_rows"] += int(mask.sum())
        mask &= chunk["Type"].eq("single nucleotide variant")
        counters["snv_rows"] += int(mask.sum())
        review_status = chunk["ReviewStatus"].fillna("").astype(str)
        trusted_review = review_status.str.contains(TRUSTED_REVIEW_PATTERN, case=False, regex=True)
        low_confidence = review_status.str.fullmatch(LOW_CONFIDENCE_REVIEW, case=False)
        mask &= trusted_review & ~low_confidence
        counters["trusted_rows"] += int(mask.sum())
        last_eval = pd.to_datetime(chunk["LastEvaluated"], errors="coerce")
        mask &= last_eval.notna()
        counters["last_evaluated_rows"] += int(mask.sum())
        ref = chunk["ReferenceAlleleVCF"].map(clean_base)
        alt = chunk["AlternateAlleleVCF"].map(clean_base)
        position = pd.to_numeric(chunk["PositionVCF"], errors="coerce")
        mask &= ref.str.fullmatch("[ACGT]") & alt.str.fullmatch("[ACGT]") & position.notna()
        counters["basic_allele_position_rows"] += int(mask.sum())
        out = chunk.loc[mask].copy()
        if out.empty:
            continue
        out["ReferenceAlleleVCF"] = ref.loc[mask].to_numpy()
        out["AlternateAlleleVCF"] = alt.loc[mask].to_numpy()
        out["PositionVCF"] = position.loc[mask].astype(np.int64).to_numpy()
        out["Y"] = out["Y"].astype(np.int8)
        out["Chromosome"] = out["Chromosome"].map(normalize_chromosome)
        out["ChromosomeFASTA"] = out.apply(choose_chromosome_fasta, axis=1, fasta_names=fasta_names)
        out = out[out["ChromosomeFASTA"].isin(fasta_names)].copy()
        counters["chromosome_fasta_rows"] += len(out)
        if out.empty:
            continue
        reference_bases = []
        matches = []
        for row in out.itertuples(index=False):
            base = str(genome[str(row.ChromosomeFASTA)][int(row.PositionVCF) - 1:int(row.PositionVCF)]).upper()
            reference_bases.append(base)
            matches.append(base == row.ReferenceAlleleVCF)
        out["ReferenceBaseGRCh38"] = reference_bases
        out["ReferenceAlleleMatchesGRCh38"] = matches
        out = out[out["ReferenceAlleleMatchesGRCh38"]].copy()
        counters["ref_match_rows"] += len(out)
        if out.empty:
            continue
        ordered_cols = [
            "Type", "GeneSymbol", "ClinicalSignificance", "LastEvaluated", "Assembly", "Chromosome",
            "ReviewStatus", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "Y",
            "ChromosomeFASTA", "ReferenceBaseGRCh38", "ReferenceAlleleMatchesGRCh38",
        ]
        kept_chunks.append(out[ordered_cols])

    if not kept_chunks:
        raise RuntimeError("No rows passed filtering.")
    filtered = pd.concat(kept_chunks, ignore_index=True)
    filtered.to_parquet(FILTERED_PATH, index=False)
    print("Counters:", dict(counters))
    print("Saved:", FILTERED_PATH)
    print("Shape:", filtered.shape)
    print("Labels:", dict(zip(*np.unique(filtered["Y"], return_counts=True))))
    return filtered

filtered = filter_raw_clinvar()
display(filtered.head())


In [ ]:
def encode_sequence(sequence):
    encoded = BASE_TO_INDEX[np.frombuffer(sequence.encode("ascii"), dtype=np.uint8)]
    if np.any(encoded == 255):
        raise ValueError("sequence contains non-ACGT bases")
    return encoded


def build_sparse_alt_dataset():
    if METADATA_PATH.exists() and X_PATH.exists() and Y_PATH.exists() and not RUN_BUILD_DATASET:
        print("Use existing sparse-alt dataset cache")
        metadata = pd.read_parquet(METADATA_PATH)
        return metadata, np.load(X_PATH, mmap_mode="r"), np.load(Y_PATH, mmap_mode="r")

    df = pd.read_parquet(FILTERED_PATH).copy()
    genome = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True, rebuild=False)
    metadata_rows = []
    ref_indices = []
    y_values = []
    skip_counts = Counter()

    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"Extract sparse-alt {LENGTH}bp"):
        ref = clean_base(row.ReferenceAlleleVCF)
        alt = clean_base(row.AlternateAlleleVCF)
        chrom = str(row.ChromosomeFASTA)
        if chrom not in genome:
            skip_counts["chrom_missing"] += 1
            continue
        pos = int(row.PositionVCF)
        start_1based = pos - FLANK
        end_1based = pos + FLANK
        if start_1based < 1:
            skip_counts["edge_or_short"] += 1
            continue
        seq = str(genome[chrom][start_1based - 1:end_1based])
        if len(seq) != LENGTH:
            skip_counts["edge_or_short"] += 1
            continue
        if seq[FLANK] != ref:
            skip_counts["ref_mismatch"] += 1
            continue
        try:
            ref_encoded = encode_sequence(seq)
        except ValueError:
            skip_counts["non_acgt_context"] += 1
            continue
        alt_index = int(BASE_TO_INDEX[ord(alt)])
        if alt_index < 0 or alt_index > 3:
            skip_counts["bad_alt"] += 1
            continue
        record = row._asdict()
        record["ContextStart1Based"] = start_1based
        record["ContextEnd1Based"] = end_1based
        record["RefCenterIndex"] = int(ref_encoded[FLANK])
        record["AltIndex"] = alt_index
        metadata_rows.append(record)
        ref_indices.append(ref_encoded.astype(np.uint8, copy=False))
        y_values.append(int(row.Y))

    metadata = pd.DataFrame(metadata_rows)
    y = np.asarray(y_values, dtype=np.int8)
    n_rows = len(metadata)
    metadata.to_parquet(METADATA_PATH, index=False)
    np.save(Y_PATH, y)
    x = np.lib.format.open_memmap(X_PATH, mode="w+", dtype=np.uint8, shape=(n_rows, LENGTH))
    for i in tqdm(range(n_rows), desc="Write REF index tensor"):
        x[i, :] = ref_indices[i]
    x.flush()
    x = np.load(X_PATH, mmap_mode="r")
    print("Saved metadata:", METADATA_PATH, metadata.shape)
    print("Saved X_ref_index:", X_PATH, x.shape, x.dtype)
    print("Saved y:", Y_PATH, y.shape, y.dtype)
    print("Skipped:", dict(skip_counts))
    print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))
    assert "LastEvaluated" in metadata.columns
    assert "AltIndex" in metadata.columns
    return metadata, x, y

metadata, x, y = build_sparse_alt_dataset()
ALT_CENTER_INDEX = metadata["AltIndex"].to_numpy(dtype=np.int64)
print("metadata:", metadata.shape)
print("X_ref_index:", x.shape, x.dtype)
print("y:", y.shape, y.dtype)


In [ ]:
def select_label_indices(meta, label_mode):
    if label_mode == "all":
        return np.arange(len(meta), dtype=np.int64)
    if label_mode == "definitive_only":
        clinical = meta["ClinicalSignificance"].fillna("").astype(str).str.strip()
        return np.flatnonzero(clinical.isin(["Benign", "Pathogenic"]).to_numpy()).astype(np.int64)
    raise ValueError(f"Unsupported LABEL_MODE: {label_mode}")


def make_groups(meta):
    groups = meta["GeneSymbol"].fillna("unknown").astype(str).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_row_", row_ids[unknown])
    return groups


def make_chromosome_groups(meta):
    groups = meta["Chromosome"].fillna("unknown").astype(str).str.replace("chr", "", case=False, regex=False).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_chr_row_", row_ids[unknown])
    return groups


def make_genome_block_groups(meta, chromosome_groups, block_size_bp):
    positions = pd.to_numeric(meta["PositionVCF"], errors="coerce")
    blocks = ((positions - 1) // block_size_bp).astype("Int64")
    chrom = pd.Series(chromosome_groups, index=meta.index).astype(str)
    block_groups = chrom + ":block_" + blocks.astype(str)
    invalid = positions.isna() | chrom.isin(["", "unknown", "nan"]) | blocks.isna()
    if invalid.any():
        row_ids = pd.Series(np.arange(len(meta)), index=meta.index).astype(str)
        block_groups.loc[invalid] = "unknown_block_row_" + row_ids.loc[invalid]
    return block_groups.to_numpy(dtype=str, copy=True)


def make_group_split(labels, groups, random_state):
    all_idx = np.arange(len(labels))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_state)
    train_idx, temp_idx = next(gss1.split(all_idx, labels, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_state)
    val_rel, test_rel = next(gss2.split(temp_idx, labels[temp_idx], groups=groups[temp_idx]))
    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def maybe_subsample(indices, max_rows, labels, seed):
    if max_rows is None or len(indices) <= max_rows:
        return indices
    rng = np.random.default_rng(seed)
    selected = []
    for label in [0, 1]:
        label_idx = indices[labels[indices] == label]
        n_label = int(round(max_rows * len(label_idx) / len(indices)))
        n_label = min(len(label_idx), max(1, n_label))
        selected.append(rng.choice(label_idx, size=n_label, replace=False))
    out = np.concatenate(selected)
    rng.shuffle(out)
    return out


def nearest_distance_to_reference(meta, query_idx, reference_idx):
    ref_pos_by_chr = {}
    for chrom, sub in meta.iloc[reference_idx].groupby("Chromosome", sort=False):
        positions = pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna()
        ref_pos_by_chr[str(chrom)] = np.sort(positions.to_numpy(dtype=np.int64))
    query = meta.iloc[query_idx]
    distances = np.full(len(query_idx), np.inf, dtype=np.float64)
    for i, (chrom, pos) in enumerate(zip(query["Chromosome"].astype(str), pd.to_numeric(query["PositionVCF"], errors="coerce"))):
        if pd.isna(pos):
            continue
        ref_positions = ref_pos_by_chr.get(str(chrom))
        if ref_positions is None or len(ref_positions) == 0:
            continue
        pos_int = int(pos)
        insert_at = np.searchsorted(ref_positions, pos_int)
        best = np.inf
        if insert_at > 0:
            best = min(best, abs(pos_int - int(ref_positions[insert_at - 1])))
        if insert_at < len(ref_positions):
            best = min(best, abs(pos_int - int(ref_positions[insert_at])))
        distances[i] = best
    return distances


def distance_summary(distances):
    finite = distances[np.isfinite(distances)]
    if len(finite) == 0:
        return {"min": float("inf"), "p5": float("inf"), "median": float("inf"), "p95": float("inf")}
    return {"min": float(np.min(finite)), "p5": float(np.percentile(finite, 5)), "median": float(np.median(finite)), "p95": float(np.percentile(finite, 95))}


def variant_keys(meta, indices):
    sub = meta.iloc[indices]
    return (
        sub["Chromosome"].astype(str)
        + ":" + pd.to_numeric(sub["PositionVCF"], errors="coerce").astype("Int64").astype(str)
        + ":" + sub["ReferenceAlleleVCF"].astype(str)
        + ":" + sub["AlternateAlleleVCF"].astype(str)
    ).to_numpy(dtype=str)


def purge_exact_variant_splits(meta, train_idx, val_idx, test_idx):
    train_keys = set(variant_keys(meta, train_idx))
    keep_val = np.array([key not in train_keys for key in variant_keys(meta, val_idx)], dtype=bool)
    kept_val = val_idx[keep_val]
    reference_keys = train_keys | set(variant_keys(meta, kept_val))
    keep_test = np.array([key not in reference_keys for key in variant_keys(meta, test_idx)], dtype=bool)
    kept_test = test_idx[keep_test]
    return kept_val, kept_test, {
        "enabled": True,
        "val_rows_before_exact_variant_purge": int(len(val_idx)),
        "val_rows_after_exact_variant_purge": int(len(kept_val)),
        "val_rows_purged_by_exact_variant": int((~keep_val).sum()),
        "test_rows_before_exact_variant_purge": int(len(test_idx)),
        "test_rows_after_exact_variant_purge": int(len(kept_test)),
        "test_rows_purged_by_exact_variant": int((~keep_test).sum()),
    }


def purge_nearby_splits(meta, train_idx, val_idx, test_idx, purge_distance_bp):
    val_dist = nearest_distance_to_reference(meta, val_idx, train_idx)
    keep_val = val_dist >= purge_distance_bp
    kept_val = val_idx[keep_val]
    test_ref = np.concatenate([train_idx, kept_val])
    test_dist = nearest_distance_to_reference(meta, test_idx, test_ref)
    keep_test = test_dist >= purge_distance_bp
    kept_test = test_idx[keep_test]
    return kept_val, kept_test, {
        "val_rows_before_purge": int(len(val_idx)),
        "val_rows_after_purge": int(len(kept_val)),
        "val_rows_purged": int((~keep_val).sum()),
        "test_rows_before_purge": int(len(test_idx)),
        "test_rows_after_purge": int(len(kept_test)),
        "test_rows_purged": int((~keep_test).sum()),
        "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_val, train_idx)),
        "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_test, test_ref)),
    }


def context_hash(global_idx, mode):
    if mode == "exact_ref":
        arr = np.ascontiguousarray(x[int(global_idx)])
    elif mode == "exact_tensor":
        arr = np.ascontiguousarray(make_sparse_alt_array(int(global_idx)))
    else:
        raise ValueError(f"Unsupported sequence purge mode: {mode}")
    return hashlib.blake2b(arr.view(np.uint8), digest_size=16).digest()


def purge_sequence_context_splits(train_global, val_global, test_global, mode):
    if mode == "none":
        return val_global, test_global, {"sequence_purge_mode": mode}
    train_hashes = {context_hash(i, mode) for i in tqdm(train_global, desc="hash train contexts", leave=False)}
    keep_val = np.array([context_hash(i, mode) not in train_hashes for i in tqdm(val_global, desc="purge val sequence duplicates", leave=False)])
    kept_val = val_global[keep_val]
    val_hashes = {context_hash(i, mode) for i in tqdm(kept_val, desc="hash kept val contexts", leave=False)}
    reference_hashes = train_hashes | val_hashes
    keep_test = np.array([context_hash(i, mode) not in reference_hashes for i in tqdm(test_global, desc="purge test sequence duplicates", leave=False)])
    kept_test = test_global[keep_test]
    return kept_val, kept_test, {
        "sequence_purge_mode": mode,
        "val_rows_before_sequence_purge": int(len(val_global)),
        "val_rows_after_sequence_purge": int(len(kept_val)),
        "val_rows_purged_by_sequence": int((~keep_val).sum()),
        "test_rows_before_sequence_purge": int(len(test_global)),
        "test_rows_after_sequence_purge": int(len(kept_test)),
        "test_rows_purged_by_sequence": int((~keep_test).sum()),
    }


def label_count_dict(labels, indices):
    values, counts = np.unique(labels[indices], return_counts=True)
    return {int(k): int(v) for k, v in zip(values, counts)}


def prevalence(labels, indices):
    return float(np.mean(labels[indices].astype(np.float64))) if len(indices) else float("nan")


def year_summary(years, indices):
    if years is None or len(indices) == 0:
        return {"min": None, "max": None, "counts": {}}
    values = years.iloc[indices].dropna().astype(int)
    if values.empty:
        return {"min": None, "max": None, "counts": {}}
    return {"min": int(values.min()), "max": int(values.max()), "counts": {int(k): int(v) for k, v in values.value_counts().sort_index().items()}}


def split_by_time(years):
    all_idx = np.arange(len(years), dtype=np.int64)
    values = years.to_numpy()
    return (
        all_idx[values <= TEMPORAL_TRAIN_MAX_YEAR],
        all_idx[values == TEMPORAL_VAL_YEAR],
        all_idx[values >= TEMPORAL_TEST_MIN_YEAR],
    )


def split_by_space_time(labels, block_groups, years):
    block_train, block_val, block_test = make_group_split(labels, block_groups, RANDOM_STATE)
    train_blocks = set(block_groups[block_train])
    val_blocks = set(block_groups[block_val])
    test_blocks = set(block_groups[block_test])
    values = years.to_numpy()
    all_idx = np.arange(len(labels), dtype=np.int64)
    return (
        all_idx[np.isin(block_groups, list(train_blocks)) & (values <= TEMPORAL_TRAIN_MAX_YEAR)],
        all_idx[np.isin(block_groups, list(val_blocks)) & (values == TEMPORAL_VAL_YEAR)],
        all_idx[np.isin(block_groups, list(test_blocks)) & (values >= TEMPORAL_TEST_MIN_YEAR)],
    )


def make_split(split_mode, active_idx, meta_for_split, labels):
    years = pd.to_datetime(meta_for_split["LastEvaluated"], errors="coerce").dt.year
    keep = years.notna().to_numpy() if split_mode in {"time_only", "space_time_block", "space_only_matched"} else np.ones(len(meta_for_split), dtype=bool)
    active_idx = active_idx[keep]
    meta_for_split = meta_for_split.loc[keep].reset_index(drop=True)
    labels = labels[keep]
    years = years.loc[keep].reset_index(drop=True) if years is not None else None

    chrom_groups = make_chromosome_groups(meta_for_split)
    block_groups = make_genome_block_groups(meta_for_split, chrom_groups, GENOME_BLOCK_SIZE_BP)
    gene_groups = make_groups(meta_for_split)

    if split_mode == "time_only":
        train_local, val_local, test_local = split_by_time(years)
    elif split_mode == "space_time_block":
        train_local, val_local, test_local = split_by_space_time(labels, block_groups, years)
    elif split_mode == "space_only_matched":
        train_local, val_local, test_local = make_group_split(labels, block_groups, RANDOM_STATE)
        target_rows = SPACE_ONLY_TARGET_TRAIN_ROWS
        if target_rows is None and MATCH_SPACE_ONLY_TRAIN_SIZE:
            target_rows = len(split_by_space_time(labels, block_groups, years)[0])
        if target_rows is not None:
            train_local = maybe_subsample(train_local, target_rows, labels, RANDOM_STATE + 17)
    elif split_mode == "genome_block":
        train_local, val_local, test_local = make_group_split(labels, block_groups, RANDOM_STATE)
    elif split_mode == "chromosome":
        train_local, val_local, test_local = make_group_split(labels, chrom_groups, RANDOM_STATE)
    elif split_mode in {"gene_group", "purged_gene_group"}:
        train_local, val_local, test_local = make_group_split(labels, gene_groups, RANDOM_STATE)
    else:
        raise ValueError(f"Unsupported split_mode: {split_mode}")

    if min(len(train_local), len(val_local), len(test_local)) == 0:
        raise ValueError(f"Empty split for {split_mode}: train={len(train_local)}, val={len(val_local)}, test={len(test_local)}")

    exact_audit = {"enabled": False}
    if EXACT_VARIANT_PURGE:
        val_local, test_local, exact_audit = purge_exact_variant_splits(meta_for_split, train_local, val_local, test_local)

    coord_audit = {"enabled": False}
    coordinate_purge_modes = {"purged_gene_group", "genome_block", "space_time_block", "space_only_matched"}
    if APPLY_COORDINATE_PURGE_TO_TIME_ONLY:
        coordinate_purge_modes.add("time_only")
    if split_mode in coordinate_purge_modes:
        val_local, test_local, coord_audit = purge_nearby_splits(meta_for_split, train_local, val_local, test_local, PURGE_DISTANCE_BP)

    train_global = active_idx[train_local]
    val_global = active_idx[val_local]
    test_global = active_idx[test_local]
    kept_val_global, kept_test_global, seq_audit = purge_sequence_context_splits(train_global, val_global, test_global, SEQUENCE_PURGE_MODE)
    global_to_local = np.full(len(metadata), -1, dtype=np.int64)
    global_to_local[active_idx] = np.arange(len(active_idx), dtype=np.int64)
    val_local = global_to_local[kept_val_global]
    test_local = global_to_local[kept_test_global]

    train_idx = active_idx[train_local]
    val_idx = active_idx[val_local]
    test_idx = active_idx[test_local]
    train_blocks, val_blocks, test_blocks = set(block_groups[train_local]), set(block_groups[val_local]), set(block_groups[test_local])
    audit = {
        "split_mode": split_mode,
        "genome_block_size_bp": GENOME_BLOCK_SIZE_BP,
        "purge_distance_bp": PURGE_DISTANCE_BP,
        "sequence_purge_mode": SEQUENCE_PURGE_MODE,
        "temporal_train_max_year": TEMPORAL_TRAIN_MAX_YEAR,
        "temporal_val_year": TEMPORAL_VAL_YEAR,
        "temporal_test_min_year": TEMPORAL_TEST_MIN_YEAR,
        "selected_rows": int(len(active_idx)),
        "train_rows": int(len(train_idx)), "val_rows": int(len(val_idx)), "test_rows": int(len(test_idx)),
        "train_label_counts": label_count_dict(labels, train_local),
        "val_label_counts": label_count_dict(labels, val_local),
        "test_label_counts": label_count_dict(labels, test_local),
        "train_prevalence": prevalence(labels, train_local),
        "val_prevalence": prevalence(labels, val_local),
        "test_prevalence": prevalence(labels, test_local),
        "train_year_summary": year_summary(years, train_local),
        "val_year_summary": year_summary(years, val_local),
        "test_year_summary": year_summary(years, test_local),
        "train_block_count": int(len(train_blocks)), "val_block_count": int(len(val_blocks)), "test_block_count": int(len(test_blocks)),
        "genome_block_overlap_train_val": int(len(train_blocks & val_blocks)),
        "genome_block_overlap_train_test": int(len(train_blocks & test_blocks)),
        "genome_block_overlap_val_test": int(len(val_blocks & test_blocks)),
        "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(meta_for_split, val_local, train_local)),
        "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(meta_for_split, test_local, np.concatenate([train_local, val_local]))),
        "exact_variant_purge_audit": exact_audit,
        "coordinate_purge_audit": coord_audit,
        "sequence_purge_audit": seq_audit,
    }
    return train_idx, val_idx, test_idx, audit

active_idx = select_label_indices(metadata, LABEL_MODE)
metadata_for_split = metadata.iloc[active_idx].reset_index(drop=True)
y_for_split = np.asarray(y[active_idx], dtype=np.int8)
print("active rows:", f"{len(active_idx):,}", "labels:", dict(zip(*np.unique(y_for_split, return_counts=True))))


In [ ]:
# === TRUE two-snapshot temporal split ===
def build_old_variant_keys():
    keys = set()
    usecols = {"Assembly", "Type", "Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"}
    reader = pd.read_csv(OLD_SNAPSHOT_PATH, sep="\t", dtype=str, low_memory=False,
                         chunksize=500_000, usecols=lambda c: c in usecols)
    for chunk in tqdm(reader, desc="Scan OLD snapshot for variant keys"):
        chunk = chunk[(chunk["Assembly"] == "GRCh38") & (chunk["Type"] == "single nucleotide variant")]
        if chunk.empty:
            continue
        chrom = chunk["Chromosome"].map(normalize_chromosome).astype(str)
        pos = pd.to_numeric(chunk["PositionVCF"], errors="coerce").astype("Int64").astype(str)
        ref = chunk["ReferenceAlleleVCF"].map(clean_base)
        alt = chunk["AlternateAlleleVCF"].map(clean_base)
        keys.update((chrom + ":" + pos + ":" + ref + ":" + alt).tolist())
    return keys

OLD_KEYS = build_old_variant_keys()
print("OLD snapshot variant keys:", f"{len(OLD_KEYS):,}")


def make_two_snapshot_split(active_idx, meta_for_split, labels):
    all_local = np.arange(len(meta_for_split), dtype=np.int64)
    keys = variant_keys(meta_for_split, all_local)
    in_old = np.array([k in OLD_KEYS for k in keys], dtype=bool)
    old_local = all_local[in_old]
    new_local = all_local[~in_old]
    if len(old_local) == 0 or len(new_local) == 0:
        raise ValueError(f"two_snapshot split empty: old={len(old_local)}, new={len(new_local)}")

    chrom_groups = make_chromosome_groups(meta_for_split)
    block_groups = make_genome_block_groups(meta_for_split, chrom_groups, GENOME_BLOCK_SIZE_BP)
    gene_groups = make_groups(meta_for_split)
    dev_groups = block_groups if TWO_SNAPSHOT_DEV_SPLIT == "genome_block" else gene_groups

    gss = GroupShuffleSplit(n_splits=1, test_size=TWO_SNAPSHOT_VAL_FRACTION, random_state=RANDOM_STATE)
    tr_rel, val_rel = next(gss.split(old_local, labels[old_local], groups=dev_groups[old_local]))
    train_local = old_local[tr_rel]
    val_local = old_local[val_rel]
    test_local = new_local.copy()

    # Type-1: exact-variant purge cho cả val và test (bắt buộc).
    exact_audit = {"enabled": False}
    if EXACT_VARIANT_PURGE:
        val_local, test_local, exact_audit = purge_exact_variant_splits(meta_for_split, train_local, val_local, test_local)

    # Coordinate purge: chỉ VAL vs TRAIN (giữ TEST realistic, proximity báo cáo dạng stratum).
    val_dist = nearest_distance_to_reference(meta_for_split, val_local, train_local)
    val_local = val_local[val_dist >= PURGE_DISTANCE_BP]

    # Sequence purge (exact REF context): chỉ VAL vs TRAIN.
    seq_audit = {"sequence_purge_mode": SEQUENCE_PURGE_MODE, "applied_to": "val_only"}
    if SEQUENCE_PURGE_MODE != "none":
        train_hashes = {context_hash(int(active_idx[i]), SEQUENCE_PURGE_MODE) for i in tqdm(train_local, desc="hash train ctx", leave=False)}
        keep_val = np.array([context_hash(int(active_idx[i]), SEQUENCE_PURGE_MODE) not in train_hashes for i in tqdm(val_local, desc="purge val ctx", leave=False)])
        seq_audit["val_rows_before"] = int(len(val_local)); seq_audit["val_rows_after"] = int(keep_val.sum())
        val_local = val_local[keep_val]

    years = pd.to_datetime(meta_for_split["LastEvaluated"], errors="coerce").dt.year
    train_idx = active_idx[train_local]
    val_idx = active_idx[val_local]
    test_idx = active_idx[test_local]
    train_blocks, val_blocks, test_blocks = set(block_groups[train_local]), set(block_groups[val_local]), set(block_groups[test_local])
    audit = {
        "split_mode": "two_snapshot_temporal",
        "old_snapshot": OLD_SNAPSHOT_NAME,
        "new_snapshot": NEW_SNAPSHOT_NAME,
        "old_snapshot_path": str(OLD_SNAPSHOT_PATH),
        "new_snapshot_path": str(VARIANT_SUMMARY_PATH),
        "two_snapshot_val_fraction": TWO_SNAPSHOT_VAL_FRACTION,
        "two_snapshot_dev_split": TWO_SNAPSHOT_DEV_SPLIT,
        "genome_block_size_bp": GENOME_BLOCK_SIZE_BP,
        "purge_distance_bp": PURGE_DISTANCE_BP,
        "sequence_purge_mode": SEQUENCE_PURGE_MODE,
        "selected_rows": int(len(active_idx)),
        "rows_in_old": int(in_old.sum()), "rows_new": int((~in_old).sum()),
        "train_rows": int(len(train_idx)), "val_rows": int(len(val_idx)), "test_rows": int(len(test_idx)),
        "train_label_counts": label_count_dict(labels, train_local),
        "val_label_counts": label_count_dict(labels, val_local),
        "test_label_counts": label_count_dict(labels, test_local),
        "train_prevalence": prevalence(labels, train_local),
        "val_prevalence": prevalence(labels, val_local),
        "test_prevalence": prevalence(labels, test_local),
        "train_year_summary": year_summary(years, train_local),
        "val_year_summary": year_summary(years, val_local),
        "test_year_summary": year_summary(years, test_local),
        "genome_block_overlap_train_val": int(len(train_blocks & val_blocks)),
        "genome_block_overlap_train_test": int(len(train_blocks & test_blocks)),
        "exact_variant_purge_audit": exact_audit,
        "sequence_purge_audit": seq_audit,
    }
    return train_idx, val_idx, test_idx, audit


_orig_make_split = make_split
def make_split(split_mode, active_idx, meta_for_split, labels):
    if split_mode == "two_snapshot_temporal":
        return make_two_snapshot_split(active_idx, meta_for_split, labels)
    return _orig_make_split(split_mode, active_idx, meta_for_split, labels)

print("two_snapshot_temporal split sẵn sàng.")

In [ ]:
INPUT_CHANNELS = 8
POSITION_INDEX = np.arange(LENGTH, dtype=np.int64)


def make_sparse_alt_array(global_idx):
    ref_idx = np.asarray(x[int(global_idx)], dtype=np.int64)
    arr = np.zeros((LENGTH, 8), dtype=np.float32)
    arr[POSITION_INDEX, ref_idx] = 1.0
    arr[FLANK, 4 + int(ALT_CENTER_INDEX[int(global_idx)])] = 1.0
    return arr


def reverse_complement_8ch(arr):
    out = arr[::-1].copy()
    ref = out[:, :4].copy()
    alt = out[:, 4:8].copy()
    out[:, :4] = ref[:, COMPLEMENT]
    out[:, 4:8] = alt[:, COMPLEMENT]
    return out


class ClinVarDataset(Dataset):
    def __init__(self, indices, random_reverse_complement=False, reverse_complement=False):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.random_reverse_complement = random_reverse_complement
        self.reverse_complement = reverse_complement

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        global_idx = int(self.indices[idx])
        arr = make_sparse_alt_array(global_idx)
        if self.reverse_complement or (self.random_reverse_complement and random.random() < 0.5):
            arr = reverse_complement_8ch(arr)
        return torch.from_numpy(arr.T.copy()), torch.tensor(float(y[global_idx]), dtype=torch.float32)


def make_loader(indices, batch_size, shuffle=False, sampler=None, random_reverse_complement=False, reverse_complement=False):
    kwargs = dict(batch_size=batch_size, shuffle=shuffle if sampler is None else False, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    if NUM_WORKERS > 0:
        kwargs.update(prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS)
    return DataLoader(ClinVarDataset(indices, random_reverse_complement, reverse_complement), **kwargs)


def make_weighted_sampler(indices):
    labels = np.asarray(y[indices], dtype=np.int64)
    if IMBALANCE_STRATEGY != "weighted_sampler":
        return None
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    weights = 1.0 / np.maximum(counts, 1.0)
    sample_weights = weights[labels]
    return WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)


def compute_pos_weight(indices):
    labels = np.asarray(y[indices], dtype=np.int64)
    pos = max(1, int((labels == 1).sum()))
    neg = max(1, int((labels == 0).sum()))
    if IMBALANCE_STRATEGY == "pos_weight":
        return float(neg / pos)
    if IMBALANCE_STRATEGY == "legacy_sqrt_sampler_pos_weight":
        return float(np.sqrt(neg / pos))
    return 1.0

print("INPUT_CHANNELS:", INPUT_CHANNELS)


In [ ]:
class DilatedClinVarCNN(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        layers = [nn.Conv1d(in_channels, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU()]
        for dilation in [1, 2, 4, 8, 16, 32, 64]:
            layers.extend([
                nn.Conv1d(64, 64, kernel_size=7, padding=3 * dilation, dilation=dilation),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(0.05),
            ])
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Linear(64 * 3, 96), nn.ReLU(), nn.Dropout(0.25), nn.Linear(96, 1))

    def forward(self, x):
        features = self.features(x)
        center = features[:, :, features.shape[-1] // 2]
        global_max = torch.amax(features, dim=-1)
        global_mean = torch.mean(features, dim=-1)
        pooled = torch.cat([center, global_max, global_mean], dim=1)
        return self.classifier(pooled).squeeze(1)


def make_model():
    return DilatedClinVarCNN(INPUT_CHANNELS)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
def safe_auc(fn, labels, probs):
    try:
        return float(fn(labels, probs))
    except ValueError:
        return float("nan")


def evaluate_probs(labels, probs):
    pred = (probs >= 0.5).astype(int)
    prevalence = float(np.mean(np.asarray(labels, dtype=np.float64))) if len(labels) else float("nan")
    pr_auc = safe_auc(average_precision_score, labels, probs)
    return {
        "accuracy_at_0_5": float(accuracy_score(labels, pred)),
        "accuracy": float(accuracy_score(labels, pred)),  # backward-compatible alias; not a primary metric.
        "roc_auc": safe_auc(roc_auc_score, labels, probs),
        "pr_auc": pr_auc,
        "prevalence": prevalence,
        "pr_auc_lift": float(pr_auc / prevalence) if prevalence > 0 else float("nan"),
    }


def threshold_table(labels, probs):
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    rows = []
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        f1 = 0.0 if p + r == 0 else 2 * p * r / (p + r)
        f2 = 0.0 if 4 * p + r == 0 else 5 * p * r / (4 * p + r)
        rows.append({"threshold": float(t), "precision": float(p), "recall": float(r), "f1": float(f1), "f2": float(f2)})
    return pd.DataFrame(rows)


def metrics_at_threshold(labels, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(labels, pred)),
        "precision_pathogenic": float(precision_score(labels, pred, zero_division=0)),
        "recall_pathogenic": float(recall_score(labels, pred, zero_division=0)),
        "f1_pathogenic": float(f1_score(labels, pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(labels, pred).tolist(),
    }


def threshold_row_or_none(table, constraint_col, min_value, optimize_col):
    candidates = table[table[constraint_col] >= min_value]
    if candidates.empty:
        return None
    return candidates.loc[candidates[optimize_col].idxmax()].to_dict()


def float_dict_or_none(row):
    if row is None:
        return None
    return {k: float(v) for k, v in row.items()}


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    probs_all, labels_all = [], []
    for xb, yb in tqdm(loader, leave=False):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            logits = model(xb)
            loss = criterion(logits, yb)
        if is_train:
            if scaler is not None and scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        losses.append(float(loss.detach().cpu()) * len(yb))
        probs_all.append(torch.sigmoid(logits.detach()).cpu().numpy())
        labels_all.append(yb.detach().cpu().numpy())
    labels = np.concatenate(labels_all).astype(int)
    probs = np.concatenate(probs_all)
    metrics = evaluate_probs(labels, probs)
    metrics["loss"] = float(np.sum(losses) / len(labels))
    return metrics, probs, labels


def run_eval_tta(model, loader, rc_loader, criterion):
    metrics, probs, labels = run_epoch(model, loader, criterion)
    original_loss = metrics["loss"]
    if rc_loader is not None:
        _, rc_probs, _ = run_epoch(model, rc_loader, criterion)
        probs = (probs + rc_probs) / 2.0
        metrics = evaluate_probs(labels, probs)
        # Loss is from original orientation only; keep it as a stable diagnostic.
        metrics["loss"] = original_loss
    return metrics, probs, labels


In [ ]:
if RUN_SMOKE_TEST:
    split_mode = SPLIT_MODES[0]
    train_idx, val_idx, test_idx, split_audit = make_split(split_mode, active_idx, metadata_for_split, y_for_split)
    smoke_idx = train_idx[:8]
    xb, yb = next(iter(make_loader(smoke_idx, batch_size=8)))
    model = make_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    logits = model(xb.to(DEVICE))
    loss = criterion(logits, yb.to(DEVICE))
    loss.backward()
    print("Smoke split:", split_mode)
    print(json.dumps({k: split_audit[k] for k in ["split_mode", "train_rows", "val_rows", "test_rows", "train_year_summary", "val_year_summary", "test_year_summary", "genome_block_overlap_train_test"]}, indent=2))
    print("input:", tuple(xb.shape), "labels:", tuple(yb.shape), "logits:", tuple(logits.shape), "loss:", float(loss.detach().cpu()))


In [ ]:
def train_one_split(split_mode, debug=False):
    print("\n===== SPLIT", split_mode, "debug=", debug, "=====")
    train_idx, val_idx, test_idx, split_audit = make_split(split_mode, active_idx, metadata_for_split, y_for_split)
    if debug:
        train_idx = maybe_subsample(train_idx, DEBUG_MAX_TRAIN_ROWS, np.asarray(y), RANDOM_STATE)
        val_idx = maybe_subsample(val_idx, DEBUG_MAX_VAL_ROWS, np.asarray(y), RANDOM_STATE + 1)
        test_idx = maybe_subsample(test_idx, DEBUG_MAX_TEST_ROWS, np.asarray(y), RANDOM_STATE + 2)

    run_name = f"{SAFE_BASE}_{split_mode}"
    if debug:
        run_name += "_debug"
    paths = {
        "model": MODEL_DIR / f"clinvar_{run_name}_pytorch.pt",
        "metrics": PROCESSED_DIR / f"{run_name}_metrics.json",
        "history": PROCESSED_DIR / f"{run_name}_history.parquet",
        "predictions": PROCESSED_DIR / f"{run_name}_test_predictions.parquet",
        "thresholds": PROCESSED_DIR / f"{run_name}_thresholds.parquet",
        "val_thresholds": PROCESSED_DIR / f"{run_name}_val_thresholds.parquet",
        "split_indices": PROCESSED_DIR / f"{run_name}_split_indices.npz",
    }

    train_sampler = make_weighted_sampler(train_idx)
    train_loader = make_loader(train_idx, BATCH_SIZE, sampler=train_sampler, shuffle=train_sampler is None, random_reverse_complement=RC_AUGMENT)
    train_eval_loader = make_loader(train_idx, BATCH_SIZE)
    train_eval_rc_loader = make_loader(train_idx, BATCH_SIZE, reverse_complement=True) if RC_TTA else None
    val_loader = make_loader(val_idx, BATCH_SIZE)
    val_rc_loader = make_loader(val_idx, BATCH_SIZE, reverse_complement=True) if RC_TTA else None
    test_loader = make_loader(test_idx, BATCH_SIZE)
    test_rc_loader = make_loader(test_idx, BATCH_SIZE, reverse_complement=True) if RC_TTA else None

    pos_weight_value = compute_pos_weight(train_idx)
    criterion = nn.BCEWithLogitsLoss() if pos_weight_value == 1.0 else nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
    model = make_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    history = []
    best_val_pr_auc = -np.inf
    best_state = None
    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        print(f"Epoch {epoch}/{EPOCHS}")
        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_metrics, _, _ = run_eval_tta(model, val_loader, val_rc_loader, criterion)
        row = {"phase": "main", "epoch": epoch, "train_loss": train_metrics["loss"], "val_loss": val_metrics["loss"], "val_roc_auc": val_metrics["roc_auc"], "val_pr_auc": val_metrics["pr_auc"]}
        history.append(row)
        print(row)
        if val_metrics["pr_auc"] > best_val_pr_auc:
            best_val_pr_auc = val_metrics["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    if HARD_NEGATIVE_EPOCHS > 0 and not debug:
        print("Hard-negative mining...")
        _, train_probs, train_labels = run_eval_tta(model, train_eval_loader, train_eval_rc_loader, criterion)
        pos_idx = train_idx[train_labels == 1]
        hard_neg_idx = train_idx[(train_labels == 0) & (train_probs >= 0.50)]
        easy_neg_idx = train_idx[(train_labels == 0) & (train_probs < 0.50)]
        max_easy = int((len(pos_idx) + len(hard_neg_idx)) * 1.0)
        rng = np.random.default_rng(RANDOM_STATE)
        if len(easy_neg_idx) > max_easy:
            easy_neg_idx = rng.choice(easy_neg_idx, size=max_easy, replace=False)
        hard_train_idx = np.concatenate([pos_idx, hard_neg_idx, easy_neg_idx])
        rng.shuffle(hard_train_idx)
        hard_sampler = make_weighted_sampler(hard_train_idx)
        hard_loader = make_loader(hard_train_idx, BATCH_SIZE, sampler=hard_sampler, random_reverse_complement=RC_AUGMENT)
        optimizer = torch.optim.AdamW(model.parameters(), lr=HARD_NEGATIVE_LR, weight_decay=WEIGHT_DECAY)
        for epoch in range(1, HARD_NEGATIVE_EPOCHS + 1):
            train_metrics, _, _ = run_epoch(model, hard_loader, criterion, optimizer, scaler)
            val_metrics, _, _ = run_eval_tta(model, val_loader, val_rc_loader, criterion)
            row = {"phase": "hard_negative", "epoch": epoch, "train_loss": train_metrics["loss"], "val_loss": val_metrics["loss"], "val_roc_auc": val_metrics["roc_auc"], "val_pr_auc": val_metrics["pr_auc"]}
            history.append(row)
            print(row)
            if val_metrics["pr_auc"] > best_val_pr_auc:
                best_val_pr_auc = val_metrics["pr_auc"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if best_state is not None:
            model.load_state_dict(best_state)

    final_val_metrics, val_probs, val_labels = run_eval_tta(model, val_loader, val_rc_loader, criterion)
    test_metrics, test_probs, test_labels = run_eval_tta(model, test_loader, test_rc_loader, criterion)
    val_thresholds = threshold_table(val_labels, val_probs)
    test_thresholds = threshold_table(test_labels, test_probs)
    best_val_f1 = val_thresholds.loc[val_thresholds["f1"].idxmax()].to_dict()
    best_val_f2 = val_thresholds.loc[val_thresholds["f2"].idxmax()].to_dict()
    val_recall_at_precision_80 = threshold_row_or_none(val_thresholds, "precision", 0.80, "recall")
    val_precision_at_recall_90 = threshold_row_or_none(val_thresholds, "recall", 0.90, "precision")
    val_prev = float(np.mean(val_labels.astype(np.float64)))
    test_prev = float(np.mean(test_labels.astype(np.float64)))
    predictions = metadata.iloc[test_idx].copy()
    predictions["y_true"] = test_labels
    predictions["pred_proba_pathogenic"] = test_probs
    metrics = {
        "model_name": run_name,
        "input_design": "sparse_alt_8ch_ref_onehot_plus_alt_at_center",
        "length": LENGTH,
        "input_channels": INPUT_CHANNELS,
        "split_audit": split_audit,
        "best_val_pr_auc": float(best_val_pr_auc),
        "final_val_metrics": final_val_metrics,
        "test_metrics": test_metrics,
        "val_prevalence": val_prev,
        "test_prevalence": test_prev,
        "val_pr_auc_lift": float(final_val_metrics["pr_auc"] / val_prev) if val_prev > 0 else float("nan"),
        "test_pr_auc_lift": float(test_metrics["pr_auc"] / test_prev) if test_prev > 0 else float("nan"),
        "primary_metrics_note": "Primary metrics are threshold-free PR-AUC/ROC-AUC/lift. Thresholded test metrics use thresholds selected on validation only.",
        "best_val_f1_threshold": {k: float(v) for k, v in best_val_f1.items()},
        "best_val_f2_threshold": {k: float(v) for k, v in best_val_f2.items()},
        "val_recall_at_precision_80_threshold": float_dict_or_none(val_recall_at_precision_80),
        "val_precision_at_recall_90_threshold": float_dict_or_none(val_precision_at_recall_90),
        "test_at_best_val_f1_threshold": metrics_at_threshold(test_labels, test_probs, float(best_val_f1["threshold"])),
        "test_at_best_val_f2_threshold": metrics_at_threshold(test_labels, test_probs, float(best_val_f2["threshold"])),
        "train_rows": int(len(train_idx)), "val_rows": int(len(val_idx)), "test_rows": int(len(test_idx)),
        "runtime_minutes": float((time.time() - start) / 60.0),
    }
    if val_recall_at_precision_80 is not None:
        metrics["test_at_val_recall_at_precision_80_threshold"] = metrics_at_threshold(test_labels, test_probs, float(val_recall_at_precision_80["threshold"]))
    if val_precision_at_recall_90 is not None:
        metrics["test_at_val_precision_at_recall_90_threshold"] = metrics_at_threshold(test_labels, test_probs, float(val_precision_at_recall_90["threshold"]))
    pd.DataFrame(history).to_parquet(paths["history"], index=False)
    predictions.to_parquet(paths["predictions"], index=False)
    test_thresholds.to_parquet(paths["thresholds"], index=False)
    val_thresholds.to_parquet(paths["val_thresholds"], index=False)
    np.savez_compressed(paths["split_indices"], train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
    paths["metrics"].write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    torch.save({"model_state_dict": model.state_dict(), "config": metrics}, paths["model"])
    print("Saved metrics:", paths["metrics"])
    print("Saved model:", paths["model"])
    return metrics

if RUN_MINI_TRAIN:
    _ = train_one_split(SPLIT_MODES[0], debug=True)


In [ ]:
summaries = []
if RUN_FULL_TRAIN:
    for split_mode in SPLIT_MODES:
        summaries.append(train_one_split(split_mode, debug=False))

if summaries:
    summary_df = pd.DataFrame([
        {
            "model_name": m["model_name"],
            "split_mode": m["split_audit"]["split_mode"],
            "train_rows": m["train_rows"],
            "val_rows": m["val_rows"],
            "test_rows": m["test_rows"],
            "test_prevalence": m["test_prevalence"],
            "test_pr_auc": m["test_metrics"]["pr_auc"],
            "test_pr_auc_lift": m["test_pr_auc_lift"],
            "test_roc_auc": m["test_metrics"]["roc_auc"],
            "test_f1_at_best_val_f1": m["test_at_best_val_f1_threshold"]["f1_pathogenic"],
            "precision_at_best_val_f1": m["test_at_best_val_f1_threshold"]["precision_pathogenic"],
            "recall_at_best_val_f1": m["test_at_best_val_f1_threshold"]["recall_pathogenic"],
            "test_recall_at_val_precision80": m.get("test_at_val_recall_at_precision_80_threshold", {}).get("recall_pathogenic", np.nan),
            "test_precision_at_val_recall90": m.get("test_at_val_precision_at_recall_90_threshold", {}).get("precision_pathogenic", np.nan),
        }
        for m in summaries
    ])
    display(summary_df)
    summary_path = PROCESSED_DIR / f"{SAFE_BASE}_train_test_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Saved summary:", summary_path)
else:
    print("RUN_FULL_TRAIN=False, no full summary generated.")
